# MCDM Baseline Methods — All Schemes (A, B, C, D)
**Methods:** Weighted-Sum, TOPSIS, AHP
**Note:** LambdaMART moved to dedicated notebook (lambdamart_bagging.ipynb)
**Schemes:** A (Equal), B (Latency-dominant), C (Availability-dominant), D (Throughput-dominant)
**Metrics:** MAE, Top-1 Accuracy, Spearman, NDCG

In [1]:
# CELL 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

BASE_PATH    = '/content/drive/MyDrive/Colab Notebooks/Ranking_Selection/Dataset/'
RESULTS_PATH = '/content/drive/MyDrive/Colab Notebooks/Ranking_Selection/exp_ Comparator Baselines/Results/'

import os
os.makedirs(RESULTS_PATH, exist_ok=True)
print('Drive mounted!')

Mounted at /content/drive
Drive mounted!


In [2]:
# CELL 2: Install and import libraries
!pip install scikit-learn numpy pandas scipy -q

import pandas as pd
import numpy as np
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

print('Imports done!')

Imports done!


In [3]:
# CELL 3: Define all 4 weight schemes and file paths

NORM_COLS = [
    'Response_Time_norm',
    'Availability_norm',
    'Throughput_norm',
    'Reliability_norm',
    'Latency_norm'
]

SCHEMES = {
    'A': {
        'description': 'Equal weights',
        'weights': {
            'Response_Time_norm': 0.2,
            'Availability_norm':  0.2,
            'Throughput_norm':    0.2,
            'Reliability_norm':   0.2,
            'Latency_norm':       0.2
        },
        'test_file': 'Scheme_A/obj3_test_A.csv'
    },
    'B': {
        'description': 'Latency-dominant',
        'weights': {
            'Response_Time_norm': 0.3,
            'Availability_norm':  0.1,
            'Throughput_norm':    0.1,
            'Reliability_norm':   0.1,
            'Latency_norm':       0.4
        },
        'test_file': 'Scheme_B/obj3_test_B.csv'
    },
    'C': {
        'description': 'Availability-dominant',
        'weights': {
            'Response_Time_norm': 0.0667,
            'Availability_norm':  0.5,
            'Throughput_norm':    0.0667,
            'Reliability_norm':   0.3,
            'Latency_norm':       0.0667
        },
        'test_file': 'Scheme_C/obj3_test_C.csv'
    },
    'D': {
        'description': 'Throughput-dominant',
        'weights': {
            'Response_Time_norm': 0.3,
            'Availability_norm':  0.0667,
            'Throughput_norm':    0.5,
            'Reliability_norm':   0.0667,
            'Latency_norm':       0.0667
        },
        'test_file': 'Scheme_D/obj3_test_D.csv'
    }
}

for s, info in SCHEMES.items():
    total = sum(info['weights'].values())
    print(f'Scheme {s} ({info["description"]}): weights sum = {total:.4f}')

Scheme A (Equal weights): weights sum = 1.0000
Scheme B (Latency-dominant): weights sum = 1.0000
Scheme C (Availability-dominant): weights sum = 1.0001
Scheme D (Throughput-dominant): weights sum = 1.0001


In [4]:
# CELL 4: Evaluation metric functions
# Returns aggregate metrics AND per-list Top-1 binary vector
# Binary vector required for McNemar's test in stats notebook

def compute_mae(true_ranks, pred_ranks):
    return np.mean(np.abs(np.array(true_ranks) - np.array(pred_ranks)))

def compute_top1(true_ranks, pred_ranks):
    return int(np.argmin(pred_ranks) == np.argmin(true_ranks))

def compute_spearman(true_ranks, pred_ranks):
    corr, _ = spearmanr(true_ranks, pred_ranks)
    return corr if not np.isnan(corr) else 0.0

def compute_ndcg(true_ranks, pred_ranks, k=10):
    n = len(true_ranks)
    true_ranks = np.array(true_ranks)
    pred_ranks = np.array(pred_ranks)
    relevance  = (n + 1) - true_ranks
    pred_order = np.argsort(pred_ranks)
    sorted_rel = relevance[pred_order]
    dcg  = sum(sorted_rel[i] / np.log2(i + 2) for i in range(min(k, n)))
    idcg = sum(np.sort(relevance)[::-1][i] / np.log2(i + 2) for i in range(min(k, n)))
    return dcg / idcg if idcg > 0 else 0.0

def evaluate_method(test_df, pred_col):
    """Returns aggregate metrics dict AND per-list Top-1 binary vector."""
    mae_l, top1_l, sp_l, ndcg_l = [], [], [], []
    for lid in test_df['list_id'].unique():
        g = test_df[test_df['list_id'] == lid]
        tr = g['list_rank'].values
        pr = g[pred_col].values
        mae_l.append(compute_mae(tr, pr))
        top1_l.append(compute_top1(tr, pr))
        sp_l.append(compute_spearman(tr, pr))
        ndcg_l.append(compute_ndcg(tr, pr))
    metrics = {
        'MAE':      round(np.mean(mae_l), 4),
        'Top-1':    round(np.mean(top1_l) * 100, 2),
        'Spearman': round(np.mean(sp_l), 4),
        'NDCG':     round(np.mean(ndcg_l), 4)
    }
    return metrics, np.array(top1_l)

In [5]:
# CELL 5: MCDM method functions
# All three are deterministic — single run, no seeds, no CI

# --- Weighted Sum ---
def run_weighted_sum(test_df):
    EQUAL_WEIGHTS = {
        'Response_Time_norm': 0.2,
        'Availability_norm':  0.2,
        'Throughput_norm':    0.2,
        'Reliability_norm':   0.2,
        'Latency_norm':       0.2
    }
    ranks = []
    for lid in test_df['list_id'].unique():
        g = test_df[test_df['list_id'] == lid].copy()
        scores = sum(g[col] * w for col, w in EQUAL_WEIGHTS.items())
        r = scores.rank(ascending=False, method='min').astype(int)
        ranks.extend(r.values)
    test_df = test_df.copy()
    test_df['ws_rank'] = ranks
    return evaluate_method(test_df, 'ws_rank')


# --- TOPSIS ---
# Uses scheme-specific weights — adapts to each scheme
def run_topsis(test_df, weights):
    w = np.array(list(weights.values()))
    ranks = []
    for lid in test_df['list_id'].unique():
        g = test_df[test_df['list_id'] == lid].copy()
        matrix   = g[NORM_COLS].values.astype(float)
        weighted = matrix * w
        ideal_best  = weighted.max(axis=0)
        ideal_worst = weighted.min(axis=0)
        dist_best  = np.sqrt(((weighted - ideal_best)  ** 2).sum(axis=1))
        dist_worst = np.sqrt(((weighted - ideal_worst) ** 2).sum(axis=1))
        closeness  = dist_worst / (dist_best + dist_worst + 1e-10)
        r = pd.Series(closeness).rank(ascending=False, method='min').astype(int)
        ranks.extend(r.values)
    test_df = test_df.copy()
    test_df['topsis_rank'] = ranks
    return evaluate_method(test_df, 'topsis_rank')


# --- AHP ---
# Fixed expert-derived weights from Enaam's thesis pairwise matrix
# w = [RT=0.356, AV=0.053, TH=0.130, RL=0.086, LAT=0.375], CR=0.072 < 0.1
def build_ahp_weights():
    """
    Expert-derived AHP weights from Enaam's thesis pairwise matrix.
    Order: [RT, AV, TH, RL, LAT]
    CR = 0.072 < 0.1 (acceptable consistency)
    """
    weights = np.array([0.356, 0.053, 0.130, 0.086, 0.375])
    print(f'  AHP weights: RT={weights[0]:.3f}, AV={weights[1]:.3f}, '
          f'TH={weights[2]:.3f}, RL={weights[3]:.3f}, LAT={weights[4]:.3f}')
    return weights

def run_ahp(test_df):
    w = build_ahp_weights()
    ranks = []
    for lid in test_df['list_id'].unique():
        g = test_df[test_df['list_id'] == lid].copy()
        scores = g[NORM_COLS].values.astype(float) @ w
        r = pd.Series(scores).rank(ascending=False, method='min').astype(int)
        ranks.extend(r.values)
    test_df = test_df.copy()
    test_df['ahp_rank'] = ranks
    return evaluate_method(test_df, 'ahp_rank')

In [6]:
# CELL 6: Run all MCDM methods on all schemes

all_results   = {}
all_top1_vecs = {}

for scheme, info in SCHEMES.items():
    print('='*60)
    print(f'SCHEME {scheme}: {info["description"]}')
    print('='*60)

    test_df = pd.read_csv(BASE_PATH + info['test_file'])
    print(f'Loaded: {test_df["list_id"].nunique()} test lists')

    weights = info['weights']
    scheme_results   = {}
    scheme_top1_vecs = {}

    # 1. Weighted-Sum
    print('  Running Weighted-Sum...')
    m, v = run_weighted_sum(test_df)
    scheme_results['Weighted-Sum']   = m
    scheme_top1_vecs['Weighted-Sum'] = v
    print(f'   {m}')

    # 2. TOPSIS
    print('  Running TOPSIS...')
    m, v = run_topsis(test_df, weights)
    scheme_results['TOPSIS']   = m
    scheme_top1_vecs['TOPSIS'] = v
    print(f'   {m}')

    # 3. AHP
    print('  Running AHP...')
    m, v = run_ahp(test_df)
    scheme_results['AHP']   = m
    scheme_top1_vecs['AHP'] = v
    print(f'   {m}')

    all_results[f'Scheme_{scheme}']   = scheme_results
    all_top1_vecs[f'Scheme_{scheme}'] = scheme_top1_vecs

    # Save aggregate metrics CSV
    df_scheme = pd.DataFrame(scheme_results).T
    df_scheme.index.name = 'Method'
    df_scheme.to_csv(RESULTS_PATH + f'mcdm_results_scheme_{scheme}.csv')

    # Save per-list Top-1 binary vectors CSV
    vec_df = pd.DataFrame(scheme_top1_vecs)
    vec_df.to_csv(RESULTS_PATH + f'mcdm_top1_vectors_scheme_{scheme}.csv', index=False)

    print(f'  Saved: mcdm_results_scheme_{scheme}.csv')
    print(f'  Saved: mcdm_top1_vectors_scheme_{scheme}.csv')

SCHEME A: Equal weights
Loaded: 251 test lists
  Running Weighted-Sum...
   {'MAE': np.float64(0.0), 'Top-1': np.float64(100.0), 'Spearman': np.float64(1.0), 'NDCG': np.float64(1.0)}
  Running TOPSIS...
   {'MAE': np.float64(0.4438), 'Top-1': np.float64(91.63), 'Spearman': np.float64(0.9577), 'NDCG': np.float64(0.9945)}
  Running AHP...
  AHP weights: RT=0.356, AV=0.053, TH=0.130, RL=0.086, LAT=0.375
   {'MAE': np.float64(0.8112), 'Top-1': np.float64(88.45), 'Spearman': np.float64(0.9018), 'NDCG': np.float64(0.9897)}
  Saved: mcdm_results_scheme_A.csv
  Saved: mcdm_top1_vectors_scheme_A.csv
SCHEME B: Latency-dominant
Loaded: 251 test lists
  Running Weighted-Sum...
   {'MAE': np.float64(0.549), 'Top-1': np.float64(92.83), 'Spearman': np.float64(0.9442), 'NDCG': np.float64(0.9936)}
  Running TOPSIS...
   {'MAE': np.float64(0.4486), 'Top-1': np.float64(92.03), 'Spearman': np.float64(0.9579), 'NDCG': np.float64(0.9939)}
  Running AHP...
  AHP weights: RT=0.356, AV=0.053, TH=0.130, RL=0.08

In [8]:
# CELL 7: Summary tables — Top-1 and MAE

methods      = ['Weighted-Sum', 'TOPSIS', 'AHP']
schemes_list = ['Scheme_A', 'Scheme_B', 'Scheme_C', 'Scheme_D']
COL_LABELS   = ['Scheme A (equal)', 'Scheme B (latency)', 'Scheme C (avail.)', 'Scheme D (throughput)']

def format_cell(scheme, method, metric):
    val = all_results.get(scheme, {}).get(method, {}).get(metric)
    if val is None:
        return 'pending'
    return f'{round(val, 2):.2f}'

# --- Top-1 Table ---
print('TOP-1 ACCURACY (%) — single value (deterministic methods, no CI)')
print('='*80)
top1_rows = {m: {s: format_cell(s, m, 'Top-1') for s in schemes_list} for m in methods}
top1_df = pd.DataFrame(top1_rows).T
top1_df.columns = COL_LABELS
top1_df.index.name = 'Method'
print(top1_df.to_string())
top1_df.to_csv(RESULTS_PATH + 'mcdm_summary_top1_all_schemes.csv')

# --- MAE Table ---
print('\nMAE — single value (deterministic methods, no CI)')
print('='*80)
mae_rows = {m: {s: format_cell(s, m, 'MAE') for s in schemes_list} for m in methods}
mae_df = pd.DataFrame(mae_rows).T
mae_df.columns = COL_LABELS
mae_df.index.name = 'Method'
print(mae_df.to_string())
mae_df.to_csv(RESULTS_PATH + 'mcdm_summary_mae_all_schemes.csv')

# --- Spearman Table ---
print('\nSpearman — single value (deterministic methods, no CI)')
print('='*80)
sp_rows = {m: {s: format_cell(s, m, 'Spearman') for s in schemes_list} for m in methods}
sp_df = pd.DataFrame(sp_rows).T
sp_df.columns = COL_LABELS
sp_df.index.name = 'Method'
print(sp_df.to_string())
sp_df.to_csv(RESULTS_PATH + 'mcdm_summary_spearman_all_schemes.csv')

# --- NDCG Table ---
print('\nNDCG — single value (deterministic methods, no CI)')
print('='*80)
ndcg_rows = {m: {s: format_cell(s, m, 'NDCG') for s in schemes_list} for m in methods}
ndcg_df = pd.DataFrame(ndcg_rows).T
ndcg_df.columns = COL_LABELS
ndcg_df.index.name = 'Method'
print(ndcg_df.to_string())
ndcg_df.to_csv(RESULTS_PATH + 'mcdm_summary_ndcg_all_schemes.csv')

print('\nAll MCDM results saved.')

TOP-1 ACCURACY (%) — single value (deterministic methods, no CI)
             Scheme A (equal) Scheme B (latency) Scheme C (avail.) Scheme D (throughput)
Method                                                                                  
Weighted-Sum           100.00              92.83             57.37                 82.07
TOPSIS                  91.63              92.03             82.47                 96.81
AHP                     88.45              93.23             51.00                 84.86

MAE — single value (deterministic methods, no CI)
             Scheme A (equal) Scheme B (latency) Scheme C (avail.) Scheme D (throughput)
Method                                                                                  
Weighted-Sum             0.00               0.55              1.15                  1.20
TOPSIS                   0.44               0.45              0.51                  0.38
AHP                      0.81               0.48              1.76                 